In [0]:
# ============================================================
# Notebook: 01_bronze_ingest
# Purpose : Load raw PaySim CSV data into Bronze layer as Delta
#           table without any transformations.
# ============================================================

# ------------------------------------------------------------
# 1. Import required libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# ------------------------------------------------------------
# 2. Initialize Spark session
#    (Usually already available in Databricks, but kept for clarity)
# ------------------------------------------------------------

spark = SparkSession.builder.appName("FinShield_Bronze_Ingest").getOrCreate()

# ------------------------------------------------------------
# 3. Define source file path (DBFS / Volume location)
# ------------------------------------------------------------

SOURCE_PATH = "/Volumes/data_engineering/raw_data/raw_data/paysim_transactions.csv"

# ------------------------------------------------------------
# 4. Read CSV file into DataFrame
#    - header = true → first row as column names
#    - inferSchema → auto-detect data types
# ------------------------------------------------------------

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(SOURCE_PATH)

# ------------------------------------------------------------
# 5. Validate raw data (important step)
# ------------------------------------------------------------

print("Preview of raw data:")
df.display()

print("Schema of raw data:")
df.printSchema()

# ------------------------------------------------------------
# 6. Write data to Bronze layer as Delta table
#    - mode("overwrite") used for initial load to avoid duplicates
#    - mergeSchema allows schema evolution if needed
# ------------------------------------------------------------

BRONZE_TABLE = "data_engineering.bronze_layer.finshield_bronze"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

print("Bronze table created successfully.")

# ------------------------------------------------------------
# 7. Validate Bronze table
# ------------------------------------------------------------

print("Preview of Bronze table:")
spark.table(BRONZE_TABLE).display()